In [1]:
from utils import *
import wandb

In [2]:
def itertoolsBetter(dataIter):
    while True:
        for batch in dataIter:
            yield batch

In [ ]:
# https://github.com/kreasof-ai/sigreg
def sigreg_weak_loss(x, sketch_dim=64):
    """
    Forces Covariance(x) ~ Identity.
    Matches the 2nd Moment (Spherical Cloud).
    """
    N, C = x.size()
    # 1. Sketching (Optional for C=512, but good for consistency)
    if C > sketch_dim:
        S = torch.randn(sketch_dim, C, device=x.device) / (C ** 0.5)
        x = x @ S.T  # [N, sketch_dim]
    else:
        sketch_dim = C

    # 2. Centering & Covariance
    x = x - x.mean(dim=0, keepdim=True)
    cov = (x.T @ x) / (N - 1 + 1e-6)

    # 3. Target Identity
    target = torch.eye(sketch_dim, device=x.device)

    # 4. Off-diagonal suppression + Diagonal maintenance
    return torch.norm(cov - target, p='fro')


class EmbeddingLoss(nn.Module):
    def __init__(self, temperature=0.04, alpha=0.1):
        super().__init__()
        self.temperature = temperature
        self.alpha = alpha

    def forward(self, x, y, families=None):
        B = x.shape[0]
        xNorm, yNorm = nn.functional.normalize(x, dim=-1), nn.functional.normalize(y, dim=-1)

        logits = (xNorm @ yNorm.T) / self.temperature  # [B, B]

        if families is not None:
            # True where i and j share a family — these are positives
            familyMatch = (families.unsqueeze(0) == families.unsqueeze(1))  # [B, B]
            # Diagonal is always a positive
            familyMatch.fill_diagonal_(True)

            # Soft target: uniform distribution over all positives in the row
            targets = familyMatch.float()
            targets = targets / targets.sum(dim=1, keepdim=True)  # normalize rows to sum to 1

            # Mask diagonal out of the denominator for the standard InfoNCE denominator
            # but keep all positives in the numerator
            logSoftmax = torch.log_softmax(logits, dim=1)
            loss1 = -(targets * logSoftmax).sum(dim=1).mean()

            logSoftmax2 = torch.log_softmax(logits.T, dim=1)
            loss2 = -(targets.T * logSoftmax2).sum(dim=1).mean()
        else:
            # Fall back to standard InfoNCE with integer labels
            labels = torch.arange(B, device=x.device)
            loss1 = nn.functional.cross_entropy(logits, labels)
            loss2 = nn.functional.cross_entropy(logits.T, labels)

        infoLoss = 0.5 * (loss1 + loss2)
        sigLoss = 0.5 * (sigreg_weak_loss(x) + sigreg_weak_loss(y))

        return {"total": infoLoss + self.alpha * sigLoss, "info": infoLoss, "sig": sigLoss}


class Perplexity(nn.Module):
    def __init__(self, loss):
        super().__init__()
        self.loss = loss

    def forward(self, y1, y2, names):
        log = self.loss(y1, y2, names)["info"]
        return torch.exp(log)
    

def recallAtK(x, y, families=None, k=10):
    B = x.shape[0]
    xNorm, yNorm = nn.functional.normalize(x, dim=-1), nn.functional.normalize(y, dim=-1)

    logits = xNorm @ yNorm.T

    if families is not None:
        positive = (families.unsqueeze(0) == families.unsqueeze(1))
    else:
        positive = torch.eye(B, device=x.device).bool()

    topK = logits.topk(k, dim=1).indices

    hits = positive.gather(1, topK).any(dim=1).float()

    return hits.mean.item()

In [4]:
def saveExperiment(imageModel, textModel, config, experimentName, start):
    latestPath = os.path.join("checkpoints", "finetune", "latest")
    if not os.path.exists(os.path.join("checkpoints", "finetune", "latest")):
        os.mkdir(latestPath)

    stamp = start.strftime("%Y-%m-%d %H-%M")
    timePath = os.path.join("checkpoints", "finetune", stamp)
    if not os.path.exists(timePath):
        os.mkdir(timePath)

    saveToPath(latestPath, imageModel, textModel, config, experimentName)
    saveToPath(timePath, imageModel, textModel, config, experimentName)


def saveToPath(path, imageModel, textModel, config, experimentName):
    if not os.path.exists(os.path.join(path, experimentName)):
        os.mkdir(os.path.join(path, experimentName))

    torch.save(imageModel.state_dict(), os.path.join(path, experimentName, "image.pt"))
    textModel.save_pretrained(os.path.join(path, experimentName, "text"))
    # torch.save(textModel.state_dict(), os.path.join(path, experimentName, "text.pt"))
    config.save(os.path.join(path, experimentName, "config.json"))

In [ ]:
def trainModel(config, textModel, imageModel, dataset, experimentName, start, imageConfig):
    optimizer = torch.optim.Adam(imageModel.classifier.parameters() + textModel.head.parameters(), lr=config.learningRate)

    imageModel.to(device)
    textModel.to(device)

    objective = EmbeddingLoss(temperature=0.04, alpha=0.1)
    # objective = ContrastiveLoss2()
    criterion = Perplexity(objective)

    train, test = dataset.split(dataset, batchSize=config.batchSize)

    testIter = itertoolsBetter(test)

    testHistory = []

    total = 0

    run = None

    try:
        for epoch in range(config.epochs):
            progress = 0
            averageTrainLoss = 0
            averageTestLoss = 0
            for images, targets, info, text, _ in train:
                imageModel.train()
                textModel.train()
                optimizer.zero_grad()

                imageOutputs = imageModel(images.to(device))
                textOutputs = textModel(**(text.to(device))).pooler_output
                loss = objective(imageOutputs, textOutputs, info)
                trainPerplexity = criterion(imageOutputs, textOutputs, info)

                trainLoss = loss["total"].detach().item()
                averageTrainLoss = (averageTrainLoss * progress + trainLoss) / (progress + 1)

                loss["total"].backward()
                optimizer.step()

                with torch.no_grad():
                    imageModel.eval()
                    textModel.eval()
                    images1, targets1, info1, text1, _ = next(testIter)
                    imageOutputs1 = imageModel(images1.to(device))
                    textOutputs1 = textModel(**(text1.to(device))).pooler_output
                    loss1 = objective(imageOutputs1, textOutputs1, info1)
                    testPerplexity = criterion(imageOutputs1, textOutputs1, info1)

                    testLoss = loss1["total"].detach().item()
                    averageTestLoss = (averageTestLoss * progress + testLoss) / (progress + 1)

                if run is None:
                    run = wandb.init(entity="dylanberndt123-missouri-state-university", project="Briefcase", config=config.serialize())
                
                run.log({"Train Loss": trainLoss, 
                         "Train Perplexiy": trainPerplexity.detach().item(),
                         "Train InfoNCE": loss["info"].detach().item(),
                         "Train SigREG": loss["sig"].detach().item(),
                         "Test Loss": testLoss,
                         "Test Perplexity": testPerplexity.detach().item(),
                         "Test InfoNCE": loss1["info"].detach().item(),
                         "Test SigREG": loss1["sig"].detach().item(),
                         "Train Recall@1": recallAtK(imageOutputs, textOutputs, k=1),
                         "Test Recall@1": recallAtK(imageOutputs1, textOutputs1, k=1),
                         "Train Recall@5": recallAtK(imageOutputs, textOutputs, k=5),
                         "Test Recall@5": recallAtK(imageOutputs1, textOutputs1, k=5),
                         "Train Recall@10": recallAtK(imageOutputs, textOutputs, k=10),
                         "Test Recall@10": recallAtK(imageOutputs1, textOutputs1, k=10)
                         }, step=total)

                progress += 1
                total += 1

                progressString = f"\r{epoch + 1} | {progress}/{len(train)} | {(progress / len(train)) * 100:.3f}%"

                print(f"{progressString} |  Train Loss: {averageTrainLoss:.2f} | Test Loss: {averageTestLoss:.2f}",end="")

            print(f"\rEpoch {epoch + 1} | Train Loss: {averageTrainLoss:.2f} | Test Loss: {averageTestLoss:.2f}{' ' * 50}")

            if (np.array(testHistory) < averageTestLoss).sum() >= 2:
                raise KeyboardInterrupt
            testHistory.append(averageTestLoss)

    except KeyboardInterrupt:
        saveExperiment(imageModel, textModel, imageConfig, experimentName, start)
        return imageModel, textModel

    saveExperiment(imageModel, textModel, imageConfig, experimentName, start)
    return imageModel, textModel

In [6]:
queryConfig = Config().load(os.path.join("configs", "querying.json"))

In [ ]:
imageModelNames = ["latest"]
textModels = [CLIPTextModel]
textModelNames = ["openai/clip-vit-base-patch32"]

imageModel, imageConfig = ViT.load(os.path.join("checkpoints", "pretrain", "latest"))

imageConfig.dataset.directory = "dataset"
dataset = MyFontsQueryData(imageConfig.dataset)
dataset.method = "none"

for t, textModelName in enumerate(textModelNames):
    dataset.setTokenizer(textModelName)
    textModel = textModels[t].from_pretrained(textModelName)
    cfg = AutoConfig.from_pretrained(textModelName)
    if hasattr(cfg, "hidden_size"):
        textDimension = cfg.hidden_size
    else:
        textDimension = cfg.projection_dim

    for i, imageModelName in enumerate(imageModelNames):
        if imageModelName == "CLIP":
            imageConfig.model["textProjection"] = textDimension
            imageModel = CLIPEmbedder(imageConfig.model)
        else:
            imageModel, imageConfig = ViT.load(os.path.join("checkpoints", "pretrain", "latest"))
            imageConfig.model["textProjection"] = textDimension
            currentDim = imageConfig.model.embedDim
            imageModel.classifier = nn.Sequential(
                nn.LazyLinear(textDimension),
                nn.ReLU(),
                nn.LazyLinear(textDimension)
            )

        experimentName = imageModelName + " " + textModelName.replace("/", "-")

        if hasattr(imageModel, "outputType"):
            imageModel.outputType = "pooled"

        print(f"\n{'=' * 28}\n{experimentName}\n{'=' * 28}")

        imageModel, textModel = trainModel(queryConfig, textModel, imageModel, dataset, experimentName, datetime.now(), imageConfig)

        saveExperiment(imageModel, textModel, imageConfig, experimentName, datetime.now())


Images loaded: 973401/973455
Descriptions loaded: 18801/18815
100.00% of fonts have descriptions

latest openai-clip-vit-base-patch32


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\dylan\_netrc.
wandb: Currently logged in as: dylanberndt123 (dylanberndt123-missouri-state-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


1 | 1431/3033 | 47.181% |  Train Loss: 4.70 | Test Loss: 5.06